# Tabular Data Classification

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader 
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [3]:
df = pd.read_csv('/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/riceClassification.csv')
df.head(3)

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1


In [4]:
df = df.drop(columns=['id'], axis=1)
df.dropna(inplace=True)
df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [5]:
print(df['Class'].unique())
print('-'*100)
print(df['Class'].value_counts())

[1 0]
----------------------------------------------------------------------------------------------------
Class
1    9985
0    8200
Name: count, dtype: int64


In [6]:
original_df = df.copy()

In [7]:
for column in df.columns:
    df[column] = df[column]/ df[column].max()

In [8]:
df

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0
...,...,...,...,...,...,...,...,...,...,...,...
18180,0.573262,0.811219,0.618156,0.971489,0.545785,0.757140,0.562384,0.654774,0.733291,0.744543,0.0
18181,0.742899,0.925674,0.704314,0.971683,0.709121,0.861916,0.730296,0.758107,0.708884,0.745661,0.0
18182,0.623408,0.844800,0.640916,0.972058,0.593296,0.789562,0.633098,0.673049,0.754720,0.747830,0.0
18183,0.583741,0.826356,0.623551,0.972748,0.562227,0.764030,0.555396,0.675248,0.702103,0.751874,0.0


In [9]:
X = np.array(df.iloc[:,:-1])
y = np.array(df.iloc[:,-1])

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X_test,y_test, test_size=0.5, random_state=42)

In [12]:
class dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.long).to(device)

    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [13]:
training_dataset = dataset(X_train, y_train)
validation_dataset = dataset(X_val, y_val)
test_dataset = dataset(X_test, y_test)

In [14]:
train_loader = DataLoader(training_dataset, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [15]:
h_n = 10

class Mynn(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(X.shape[1], h_n)
        self.model = nn.Linear(h_n, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self,x):
        x = self.input(x)
        x = self.model(x)
        x = self.sigmoid(x)
        return x


In [16]:
model = Mynn().to(device)

In [17]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
for data, label in train_loader:
    print(data.shape)
    print(label.shape)
    break

torch.Size([32, 10])
torch.Size([32])


In [19]:
for data, label in train_loader:
    print(data)
    print(label)
    break

tensor([[0.6724, 0.8915, 0.6596, 0.9752, 0.6417, 0.8200, 0.5642, 0.7174, 0.7165,
         0.7668],
        [0.6449, 0.8910, 0.6369, 0.9792, 0.6136, 0.8030, 0.5507, 0.7097, 0.7020,
         0.7936],
        [0.5496, 0.8443, 0.5719, 0.9850, 0.5261, 0.7413, 0.5541, 0.6697, 0.6720,
         0.8376],
        [0.9159, 0.8961, 0.8924, 0.9244, 0.8788, 0.9570, 0.7844, 0.7757, 0.8347,
         0.5697],
        [0.8444, 0.8722, 0.8409, 0.9317, 0.8073, 0.9189, 0.8792, 0.7834, 0.7546,
         0.5885],
        [0.5584, 0.8066, 0.6049, 0.9735, 0.5297, 0.7472, 0.8426, 0.6535, 0.7169,
         0.7566],
        [0.8055, 0.8355, 0.8392, 0.9224, 0.7640, 0.8975, 0.8191, 0.7257, 0.8388,
         0.5648],
        [0.4473, 0.6851, 0.5711, 0.9586, 0.4292, 0.6688, 0.7280, 0.5816, 0.7252,
         0.6806],
        [0.5061, 0.7480, 0.5981, 0.9649, 0.4860, 0.7114, 0.8206, 0.6219, 0.7176,
         0.7096],
        [0.7655, 0.7959, 0.8421, 0.9092, 0.7366, 0.8749, 0.7945, 0.7088, 0.8357,
         0.5362],
        [0

In [21]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

epochs = 10
for epoch in range(epochs):
    total_acc_train = 0
    total_loss_train = 0
    total_acc_val = 0
    total_loss_val = 0

    for data, label in train_loader:

        prediction = model(data).squeeze(1)
        loss = criterion(prediction, label)
        break


: 